### run this to populate ./data with ambigqa dataset

In [1]:
# Install required packages (run once)
!pip install -q pandas matplotlib seaborn tqdm scikit-learn

In [2]:
import json
import requests
import zipfile
from pathlib import Path
import shutil

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Setup complete!")

✅ Setup complete!


/opt/anaconda3/envs/182_proj/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Create data directory
DATA_DIR = Path("./data")
DATA_DIR.mkdir(exist_ok=True)

print(f"📁 Data directory: {DATA_DIR.absolute()}")

📁 Data directory: /Users/ericwang/Documents/school/cs182/proj/ambig-interp/data


In [4]:
def download_ambignq():
    """
    Download and extract AmbigNQ dataset.
    Note: Only train and dev splits are publicly available.
    """
    # Check if already downloaded
    expected_files = [
        DATA_DIR / "train_light.json",
        DATA_DIR / "dev_light.json"
    ]
    
    if all(f.exists() for f in expected_files):
        print("✅ Dataset already downloaded!")
        return
    
    # Download
    url = "https://nlp.cs.washington.edu/ambigqa/data/ambignq_light.zip"
    zip_path = DATA_DIR / "ambignq_light.zip"
    
    print("📥 Downloading AmbigNQ dataset...")
    response = requests.get(url, stream=True)
    total = int(response.headers.get('content-length', 0))
    
    with open(zip_path, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True) as pbar:
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            pbar.update(len(chunk))
    
    # Extract
    print("📦 Extracting files...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(DATA_DIR)
    
    # Handle nested directory structure
    nested_dir = DATA_DIR / "ambignq"
    if nested_dir.exists():
        for file in nested_dir.glob("*.json"):
            shutil.move(str(file), str(DATA_DIR / file.name))
        nested_dir.rmdir()
    
    # Clean up
    zip_path.unlink()
    
    # Verify
    if all(f.exists() for f in expected_files):
        print("✅ Download complete!")
        print(f"   - train_light.json")
        print(f"   - dev_light.json")
        print("\n⚠️  Note: test_light.json is not publicly available")
        print("   We'll create our own test split from the dev set.")
    else:
        print("⚠️  Some files missing. Contents of data directory:")
        for f in DATA_DIR.glob("*.json"):
            print(f"   - {f.name}")

# Download the dataset
download_ambignq()

✅ Dataset already downloaded!
